# 第18章：行业轮动策略实战

## 本章学习目标

- 理解行业轮动策略原理
- 实现行业因子构建
- 掌握行业配置方法
- 完成回测分析

---

## 18.1 行业轮动策略概述

行业轮动策略通过在不同行业之间进行配置，捕捉行业间的相对收益机会。

### 策略逻辑

```
行业轮动策略流程:

1. 行业分类: 将股票按行业分组
   └── 申万一级行业 / 中信行业

2. 行业信号: 生成行业层面的预测信号
   └── 行业动量、行业估值、行业景气度

3. 行业配置: 根据信号分配行业权重
   └── Top-N 行业 / 动量轮动

4. 股票选择: 在每个行业内选择优质股票
   └── 行业内选股因子
```

In [ ]:
import qlib
from qlib.data import D
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 初始化 qlib
qlib.init(
    provider_uri="~/.qlib/qlib_data/cn_data",
    region="cn",
)

print("Qlib 初始化成功")

## 18.2 行业分类

In [ ]:
# 定义行业分类（示例）
# 注意：实际使用需要真实的行业分类数据

# 申万一级行业
SW_INDUSTRIES = {
    "银行": ["SH601398", "SH601939", "SH601288", "SH600036"],
    "非银金融": ["SH601318", "SH601628", "SH600030"],
    "食品饮料": ["SH600519", "SH000858", "SH000568"],
    "医药生物": ["SH600276", "SH000538", "SH300015"],
    "电子": ["SH000725", "SH002475", "SH300124"],
    "计算机": ["SH002415", "SH300033", "SH600570"],
    "机械设备": ["SH600031", "SH000333", "SH601766"],
    "化工": ["SH600309", "SH002466", "SH000830"],
    "有色金属": ["SH601899", "SH600547", "SH002460"],
    "汽车": ["SH600104", "SH000625", "SH601238"],
}

print("申万一级行业分类（示例）:")
print(f"行业数量: {len(SW_INDUSTRIES)}")

for industry, stocks in list(SW_INDUSTRIES.items())[:5]:
    print(f"  {industry}: {len(stocks)} 只股票")

In [ ]:
# 创建股票到行业的映射
stock_to_industry = {}
for industry, stocks in SW_INDUSTRIES.items():
    for stock in stocks:
        stock_to_industry[stock] = industry

print(f"股票-行业映射数量: {len(stock_to_industry)}")

## 18.3 行业因子构建

In [ ]:
# 行业因子计算
def calculate_industry_factors(stock_data, stock_to_industry):
    """
    计算行业因子
    
    参数:
        stock_data: 股票数据
        stock_to_industry: 股票到行业的映射
    
    返回:
        行业因子 DataFrame
    """
    # 行业动量因子
    industry_momentum = {}
    
    for industry in SW_INDUSTRIES.keys():
        stocks = SW_INDUSTRIES[industry]
        # 模拟行业动量
        np.random.seed(hash(industry) % 2**32)
        industry_momentum[industry] = np.random.randn()
    
    # 行业估值因子
    industry_valuation = {}
    for industry in SW_INDUSTRIES.keys():
        np.random.seed(hash(industry + "val") % 2**32)
        industry_valuation[industry] = np.random.randn()
    
    # 行业景气度因子
    industry_sentiment = {}
    for industry in SW_INDUSTRIES.keys():
        np.random.seed(hash(industry + "sent") % 2**32)
        industry_sentiment[industry] = np.random.randn()
    
    # 汇总
    factors = pd.DataFrame({
        'momentum': industry_momentum,
        'valuation': industry_valuation,
        'sentiment': industry_sentiment,
    })
    
    # 综合评分
    factors['score'] = (
        factors['momentum'] * 0.4 + 
        factors['valuation'] * 0.3 + 
        factors['sentiment'] * 0.3
    )
    
    return factors

# 计算行业因子
industry_factors = calculate_industry_factors(None, stock_to_industry)

print("行业因子:")
industry_factors.sort_values('score', ascending=False)

In [ ]:
# 可视化行业因子
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 行业动量
industry_factors['momentum'].sort_values().plot(kind='barh', ax=axes[0, 0], color='steelblue')
axes[0, 0].set_title('行业动量因子')
axes[0, 0].set_xlabel('因子值')

# 行业估值
industry_factors['valuation'].sort_values().plot(kind='barh', ax=axes[0, 1], color='coral')
axes[0, 1].set_title('行业估值因子')
axes[0, 1].set_xlabel('因子值')

# 行业景气度
industry_factors['sentiment'].sort_values().plot(kind='barh', ax=axes[1, 0], color='green')
axes[1, 0].set_title('行业景气度因子')
axes[1, 0].set_xlabel('因子值')

# 综合评分
industry_factors['score'].sort_values().plot(kind='barh', ax=axes[1, 1], color='purple')
axes[1, 1].set_title('行业综合评分')
axes[1, 1].set_xlabel('评分')

plt.tight_layout()
plt.show()

## 18.4 行业配置策略

In [ ]:
# 行业配置策略
class SectorRotationStrategy:
    """行业轮动策略"""
    
    def __init__(self, n_sectors=5, rebalance_freq='M'):
        """
        参数:
            n_sectors: 配置的行业数量
            rebalance_freq: 再平衡频率
        """
        self.n_sectors = n_sectors
        self.rebalance_freq = rebalance_freq
    
    def get_top_sectors(self, industry_factors):
        """获取 Top-N 行业"""
        top_sectors = industry_factors.nlargest(self.n_sectors, 'score').index.tolist()
        return top_sectors
    
    def allocate_weights(self, top_sectors, industry_factors):
        """分配行业权重"""
        # 等权配置
        weights = {sector: 1.0 / self.n_sectors for sector in top_sectors}
        return weights
    
    def select_stocks_in_sector(self, sector, n_stocks=5):
        """在行业内选股"""
        stocks = SW_INDUSTRIES.get(sector, [])
        return stocks[:n_stocks]

# 创建策略
strategy = SectorRotationStrategy(n_sectors=5)

# 获取 Top 行业
top_sectors = strategy.get_top_sectors(industry_factors)
print(f"Top 5 行业: {top_sectors}")

# 分配权重
weights = strategy.allocate_weights(top_sectors, industry_factors)
print("\n行业权重:")
for sector, weight in weights.items():
    print(f"  {sector}: {weight:.2%}")

In [ ]:
# 构建股票组合
portfolio = {}
for sector in top_sectors:
    stocks = strategy.select_stocks_in_sector(sector, n_stocks=3)
    sector_weight = weights[sector]
    stock_weight = sector_weight / len(stocks)
    
    for stock in stocks:
        portfolio[stock] = {
            'sector': sector,
            'weight': stock_weight,
        }

print("股票组合:")
for stock, info in portfolio.items():
    print(f"  {stock}: 行业={info['sector']}, 权重={info['weight']:.2%}")

## 18.5 回测分析

In [ ]:
# 模拟回测
np.random.seed(42)

# 时间范围
dates = pd.date_range("2020-01-01", "2022-12-31", freq="B")
n_days = len(dates)

# 模拟各行业收益
industry_returns = {}
for industry in SW_INDUSTRIES.keys():
    np.random.seed(hash(industry) % 2**32)
    base = np.random.randn() * 0.0003
    vol = np.random.uniform(0.012, 0.018)
    industry_returns[industry] = np.random.randn(n_days) * vol + base

industry_returns_df = pd.DataFrame(industry_returns, index=dates)

print("行业收益模拟完成")
print(f"时间范围: {dates[0].date()} ~ {dates[-1].date()}")

In [ ]:
# 计算策略收益
strategy_returns = np.zeros(n_days)

for sector, weight in weights.items():
    strategy_returns += industry_returns_df[sector].values * weight

# 计算累计收益
strategy_cum = (1 + pd.Series(strategy_returns, index=dates)).cumprod()

# 基准（等权所有行业）
benchmark_returns = industry_returns_df.mean(axis=1)
benchmark_cum = (1 + benchmark_returns).cumprod()

print("回测完成")

In [ ]:
# 可视化
plt.figure(figsize=(14, 6))

plt.plot(strategy_cum.index, strategy_cum.values, label='行业轮动策略', linewidth=1.5)
plt.plot(benchmark_cum.index, benchmark_cum.values, label='基准（等权）', linewidth=1.5, linestyle='--')

plt.title('行业轮动策略 vs 基准')
plt.xlabel('日期')
plt.ylabel('累计收益')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# 绩效指标
def calculate_performance(returns):
    """计算绩效指标"""
    annual_return = (1 + returns.mean()) ** 252 - 1
    annual_vol = returns.std() * np.sqrt(252)
    sharpe = annual_return / annual_vol
    
    cum = (1 + returns).cumprod()
    running_max = cum.cummax()
    drawdown = (cum - running_max) / running_max
    max_dd = drawdown.min()
    
    return {
        "年化收益": annual_return,
        "年化波动率": annual_vol,
        "夏普比率": sharpe,
        "最大回撤": max_dd,
    }

# 计算指标
strategy_metrics = calculate_performance(pd.Series(strategy_returns))
benchmark_metrics = calculate_performance(benchmark_returns)

print("绩效对比:")
print("=" * 50)

print("\n行业轮动策略:")
for k, v in strategy_metrics.items():
    print(f"  {k}: {v:.4f}")

print("\n基准（等权）:")
for k, v in benchmark_metrics.items():
    print(f"  {k}: {v:.4f}")

## 18.6 行业收益贡献分析

In [ ]:
# 行业收益贡献
contribution = {}
for sector in top_sectors:
    sector_return = (1 + industry_returns_df[sector]).prod() - 1
    contribution[sector] = sector_return * weights[sector]

contribution_df = pd.DataFrame({
    'sector': list(contribution.keys()),
    'weight': [weights[s] for s in contribution.keys()],
    'return': [(1 + industry_returns_df[s]).prod() - 1 for s in contribution.keys()],
    'contribution': list(contribution.values()),
})

print("行业收益贡献:")
contribution_df

In [ ]:
# 可视化贡献
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 收益贡献
axes[0].bar(contribution_df['sector'], contribution_df['contribution'], color='steelblue')
axes[0].set_title('行业收益贡献')
axes[0].set_xlabel('行业')
axes[0].set_ylabel('收益贡献')
axes[0].tick_params(axis='x', rotation=45)

# 行业权重
axes[1].pie(contribution_df['weight'], labels=contribution_df['sector'], autopct='%1.1f%%')
axes[1].set_title('行业配置权重')

plt.tight_layout()
plt.show()

## 18.7 优化建议

In [ ]:
# 优化建议
print("行业轮动策略优化建议:")
print("=" * 60)

suggestions = [
    "1. 行业因子优化:",
    "   - 增加更多行业因子（如资金流向、北向持股）",
    "   - 使用机器学习方法优化因子权重",
    "",
    "2. 行业配置优化:
    "   - 考虑行业相关性进行分散化",
    "   - 添加行业动量反转信号",
    "",
    "3. 风险控制:",
    "   - 设置单一行业权重上限",
    "   - 添加止损机制",
    "",
    "4. 执行优化:",
    "   - 优化再平衡频率",
    "   - 考虑交易成本",
]

for s in suggestions:
    print(s)

## 18.8 本章小结

本章我们学习了：

1. **行业轮动策略原理**：
   - 行业分类
   - 行业因子构建
   - 行业配置

2. **行业因子构建**：
   - 行业动量因子
   - 行业估值因子
   - 行业景气度因子

3. **策略实现**：
   - Top-N 行业选择
   - 权重分配
   - 行业内选股

4. **回测分析**：
   - 策略收益
   - 行业贡献分析

### 教程总结

恭喜你完成了 Qlib 深入学习教程的全部内容！

通过本教程，你已经掌握了：
- Qlib 数据系统
- 特征工程与 Alpha 因子
- 模型训练与评估
- 策略构建与回测
- 强化学习交易
- 自定义扩展开发

### 后续学习建议

1. 深入研究 qlib 源码
2. 尝试更多模型和策略
3. 参与社区讨论
4. 进行实盘验证